# ML04 · PCA 降维

用主成分分析 principal component analysis（PCA）把高维特征压缩成少数几个方向，在保留最多信息的前提下，让数据能画成 2D 图看出结构。

## 学习目标
- 理解降维 dimensionality reduction 的动机：特征太多会带来哪些困扰。
- 创建主成分 principal component 的直觉：数据变异最大的方向。
- 知道为什么要先标准化 standardization 再做 PCA。
- 会用 sklearn 的 PCA 进行 fit 与 transform 求得降维结果。
- 会读解释变异比例 explained_variance_ratio_ 来判断该保留几维。
- 能把多维特征压到 2D 并可视化 visualization，观察群落与分布。

In [ ]:
# 概念：统一导入套件并设置画图字体，让中文标题不会变成方框
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 注意：matplotlib 预设字体不含中文，需指定一个系统有的中文字体
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "SimHei", "Arial Unicode MS"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 避免负号显示成方框

print("套件加载完成")

## 为什么要降维

降维 dimensionality reduction 是把「很多栏特征」用「比较少的栏」重新表示，尽量不丢信息。

为什么要学：特征太多时有三个困扰。
- 维度灾难 curse of dimensionality：维度越高，数据越稀疏，模型越容易过拟合 overfitting。
- 特征冗余 feature redundancy：许多栏其实彼此高度相关 correlated features，重复描述同一件事。
- 无法可视化：超过 3 维就画不出来，看不见数据结构。

何时用：字段多、彼此相关、想压缩或想画成 2D 图探索时。

In [ ]:
# 概念：用 numpy 造彼此高度相关的字段，直观感受「信息其实没那么多维」
rng = np.random.default_rng(0)   # 固定乱数种子，让每次结果一致方便对照

n = 200
# 造一个共同的「规模」因子：每栋建筑有多大，由它驱动其他字段
scale = rng.normal(loc=0.0, scale=1.0, size=n)

# 楼高、楼层数、总面积其实都跟「规模」走，只是加了一点独立杂讯
floors = 10 + 3 * scale + rng.normal(0, 0.3, n)          # 楼层数
height = 3.2 * floors + rng.normal(0, 1.0, n)            # 楼高约等于楼层数乘层高
area = 220 + 60 * scale + rng.normal(0, 5.0, n)          # 总面积也跟着规模走

data = np.column_stack([floors, height, area])           # 叠成 (200, 3) 的特征矩阵
print("数据 shape:", data.shape)

# 相关系数矩阵：值接近 1 表示两栏几乎是同一件事的不同说法
corr = np.corrcoef(data, rowvar=False)                   # rowvar=False 表示每栏是一个变量
print("三栏之间的相关系数矩阵:")
print(np.round(corr, 3))

## 主成分的直觉

主成分 principal component 就是「数据散得最开的方向」。

- 第一主成分（first principal component）：变异量 variance 最大的方向，数据沿着它拉得最长。
- 第二主成分（second principal component）：在与第一主成分垂直（正交方向 orthogonal directions）的前提下，抓剩下变异最大的方向。

把点投影 projection 到主成分上，就得到降维后的新座标。重点是直觉：找一条轴，让投影后的点尽量分得开、信息损失最少。

In [ ]:
# 概念：在 2D 散点上画出第一、第二主成分方向，看 PCA 抓到的就是最散的方向
rng = np.random.default_rng(1)

# 造一团沿着斜对角拉长的数据：楼高与面积正相关，所以呈斜椭圆分布
n = 150
base = rng.normal(0, 1, n)
height = 25 + 8 * base + rng.normal(0, 1.5, n)           # 楼高（公尺）
area = 150 + 70 * base + rng.normal(0, 12, n)            # 总面积（平方公尺）
points = np.column_stack([height, area])

# 先标准化，主成分方向才不会被面积的大数值主导（下一节详述原因）
points_std = StandardScaler().fit_transform(points)

pca = PCA(n_components=2)
pca.fit(points_std)
center = points_std.mean(axis=0)                         # 箭头从数据中心画出
# components_ 每一列是一个主成分方向；用解释变异开根号当箭头长度，方向越主要越长
lengths = np.sqrt(pca.explained_variance_)

plt.figure(figsize=(5, 5))
plt.scatter(points_std[:, 0], points_std[:, 1], s=12, alpha=0.5, label="标准化后的点")
for i, (vec, length) in enumerate(zip(pca.components_, lengths)):
    plt.arrow(center[0], center[1], vec[0] * length * 2, vec[1] * length * 2,
              width=0.03, color="red" if i == 0 else "green",
              length_includes_head=True,
              label=f"第{i + 1}主成分")
plt.axis("equal")          # 等比例座标，才看得出两轴正交
plt.legend()
plt.title("第一、第二主成分方向")
plt.xlabel("楼高（标准化）"); plt.ylabel("面积（标准化）")
plt.show()
print("两主成分是否正交（内积接近 0）:", np.round(pca.components_[0] @ pca.components_[1], 6))

## 先标准化再 PCA

PCA 看的是变异量，而变异量会受单位影响。面积动辄上万、容积率只有个位数，若不处理，PCA 几乎只会看面积这一栏。

标准化 standardization 用 z-score 把每栏转成「平均 0、标准差 1」，消除尺度差异 scale difference，让每个特征站在同一条起跑在线。sklearn 用 StandardScaler 做这件事。

公式：z = (x − 平均) / 标准差。何时用：各特征单位不同时，做 PCA 前几乎都要先标准化。

In [ ]:
# 概念：比较「未标准化」与「标准化后」第一主成分方向的差异
rng = np.random.default_rng(2)

n = 120
floors = rng.integers(3, 30, n).astype(float)            # 楼层数（个位到数十）
area = floors * 90 + rng.normal(0, 200, n) + 3000        # 面积（数千到上万，数量级远大于楼层）
raw = np.column_stack([floors, area])

# 未标准化：直接 PCA
pca_raw = PCA(n_components=2).fit(raw)
# 标准化后：先缩放再 PCA
scaled = StandardScaler().fit_transform(raw)
pca_scaled = PCA(n_components=2).fit(scaled)

# 第一主成分在 [楼层, 面积] 上的权重；数值大表示该栏主导这个方向
print("未标准化 第一主成分权重 [楼层, 面积]:", np.round(pca_raw.components_[0], 3))
print("标准化后 第一主成分权重 [楼层, 面积]:", np.round(pca_scaled.components_[0], 3))
# 注意：未标准化时面积权重几乎是 1，等于只看面积；标准化后两栏贡献才均衡
print("未标准化 解释变异比例:", np.round(pca_raw.explained_variance_ratio_, 3))
print("标准化后 解释变异比例:", np.round(pca_scaled.explained_variance_ratio_, 3))

## 用 sklearn 跑 PCA：fit 与 transform

sklearn 的 PCA 是一个估计器 estimator，用法与其他模型一致。
- fit：从数据学出主成分方向（找出那几条轴）。
- transform：把数据投影到这些轴上，得到降维后座标 transformed coordinates。
- n_components：要保留几个主成分（降到几维）。

形状：
```
pca = PCA(n_components=k)
pca.fit(X)            # 学方向，X 为 (N, D)
Z = pca.transform(X)  # 投影，Z 为 (N, k)
```
fit 与 transform 可合并成 fit_transform。

In [ ]:
# 概念：对标准化后的多栏建筑数据做 PCA(n_components=2) 的 fit 与 transform
rng = np.random.default_rng(3)

n = 100
# 造五栏建筑数据：楼高、楼层、面积、屋龄、距捷运距离
factor = rng.normal(0, 1, n)                             # 共同规模因子
height = 30 + 9 * factor + rng.normal(0, 2, n)
floors = 9 + 2.8 * factor + rng.normal(0, 0.5, n)
area = 250 + 80 * factor + rng.normal(0, 10, n)
age = rng.uniform(0, 40, n)                              # 屋龄与规模无关，较独立
dist_mrt = rng.uniform(50, 1500, n)                      # 距捷运距离，也较独立
X = np.column_stack([height, floors, area, age, dist_mrt])
print("原始特征 shape:", X.shape)

X_std = StandardScaler().fit_transform(X)                # PCA 前先标准化

pca = PCA(n_components=2)
pca.fit(X_std)                                           # 从数据学出两个主成分方向
Z = pca.transform(X_std)                                 # 投影到这两个方向
print("降维后座标 shape:", Z.shape)                        # 应为 (100, 2)
print("前三笔降维后座标:")
print(np.round(Z[:3], 3))

## 解释变异比例：该留几维

解释变异比例 explained_variance_ratio_ 告诉你每个主成分保留了原始数据多少比例的变异。

把它从大到小累加，得到累积解释变异 cumulative explained variance。常见做法是设一个门槛（例如 90%），累积到超过门槛的最少维数，就是要保留的维度数，兼顾压缩与信息保留量。

何时用：决定 n_components 要设多少时，用这条曲线来判断，而不是凭感觉。

In [ ]:
# 概念：列出各主成分解释变异比例并画累积曲线，判断压到 2 维保留多少信息
rng = np.random.default_rng(4)

n = 200
factor = rng.normal(0, 1, n)
# 六栏：前几栏受同一规模因子驱动（高度相关），后几栏较独立
f1 = factor + rng.normal(0, 0.2, n)
f2 = 2 * factor + rng.normal(0, 0.2, n)
f3 = -1.5 * factor + rng.normal(0, 0.2, n)
f4 = rng.normal(0, 1, n)
f5 = rng.normal(0, 1, n)
f6 = 0.5 * f4 + rng.normal(0, 0.3, n)
X = np.column_stack([f1, f2, f3, f4, f5, f6])

X_std = StandardScaler().fit_transform(X)
pca = PCA().fit(X_std)                                   # 不指定 n_components 会保留全部，方便看每维贡献

ratio = pca.explained_variance_ratio_
cum = np.cumsum(ratio)                                   # 从大到小累加得到累积解释变异
print("各主成分解释变异比例:", np.round(ratio, 3))
print("累积解释变异:", np.round(cum, 3))
print("压到 2 维保留的信息:", f"{cum[1]:.1%}")

plt.figure(figsize=(6, 4))
xs = np.arange(1, len(ratio) + 1)
plt.bar(xs, ratio, alpha=0.5, label="各主成分解释变异")
plt.plot(xs, cum, "o-", color="red", label="累积解释变异")
plt.axhline(0.9, color="gray", linestyle="--", label="90% 门槛")   # 门槛线辅助判断
plt.xlabel("主成分序号"); plt.ylabel("解释变异比例")
plt.title("解释变异比例与累积曲线")
plt.legend()
plt.show()

## 压到 2D 并可视化

把多维数据投影到前两个主成分后，就能用一张 2D 散点图 scatter plot 看出结构：群落、离群点、类别分离程度。

这张图的两轴是主成分而非原始特征，可看成数据的潜在空间 latent space：方向本身没有单位意义，但相对位置反映样本间的相似度。依类别上色后，若同类聚成一团、不同类分得开，代表这些特征确实能区分类别。

何时用：探索数据、检查类别是否可分、为后续聚类做准备时。

In [ ]:
# 概念：把多维建筑数据压到 2D，依用途类别（住宅／商办）上色观察是否自然分开
rng = np.random.default_rng(5)

def make_buildings(center, n, label):
    # 造同一类别的建筑：各栏围绕各自的中心抖动，仿真同类风格相近
    height = rng.normal(center[0], 3, n)
    floors = rng.normal(center[1], 1, n)
    area = rng.normal(center[2], 15, n)
    far = rng.normal(center[3], 0.3, n)                  # 容积率 floor area ratio
    X = np.column_stack([height, floors, area, far])
    y = np.full(n, label)
    return X, y

# 住宅：较矮、面积中等；商办：较高、面积大、容积率高
X_res, y_res = make_buildings([28, 9, 230, 2.2], 80, 0)   # 0 = 住宅
X_off, y_off = make_buildings([55, 18, 420, 4.5], 80, 1)  # 1 = 商办
X = np.vstack([X_res, X_off])
y = np.concatenate([y_res, y_off])

X_std = StandardScaler().fit_transform(X)
Z = PCA(n_components=2).fit_transform(X_std)             # 一步到位：学方向并投影到 2D

plt.figure(figsize=(6, 5))
for label, name, color in [(0, "住宅", "tab:blue"), (1, "商办", "tab:orange")]:
    mask = y == label                                    # 布尔遮罩挑出该类别的点
    plt.scatter(Z[mask, 0], Z[mask, 1], s=18, alpha=0.6, color=color, label=name)
plt.xlabel("第一主成分"); plt.ylabel("第二主成分")
plt.title("建筑数据压到 2D 后依用途上色")
plt.legend()
plt.show()
print("降维后 shape:", Z.shape, " 两类别是否在图上分开，可用肉眼观察")

## 练习

以下三题由浅入深，皆为复合型但简短。数据请自己用 numpy 造（仿真即可），完成后对照「验收」确认输出。

In [ ]:
# TODO 1 ·（简单）把相关特征压成一条轴（集成：相关特征 + 主成分 + fit/transform）
#   情境：你有一批建筑数据，楼高与楼层数几乎成正比，想用一个数字代表「整体规模」。
#   要求：
#     1. 用 numpy 自造高度相关的楼高与楼层数两栏数据（让楼高约等于楼层数乘层高再加杂讯）。
#     2. 对其做 PCA(n_components=1) 的 fit/transform，取得单一主成分座标。
#     3. 印出第一主成分的解释变异比例。
#   验收：应看到两栏被压成一条「规模轴」(N, 1)，且第一主成分解释了绝大部分变异（接近 1）。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）标准化前后的差异（集成：标准化 + PCA + 解释变异比例 + 可视化）
#   情境：建筑数据含面积（数值很大）与容积率（数值很小），想看尺度是否影响降维。
#   要求：
#     1. 自造含 面积、容积率、楼层数 的多栏数据（面积数千、容积率个位数）。
#     2. 分别在「未标准化」与「StandardScaler 标准化后」各跑一次 PCA 压到 2D。
#     3. 比较两者的 explained_variance_ratio_，并各画一张 2D 散点图。
#   验收：应看到未标准化时面积主导第一主成分（解释比例极高），
#         标准化后各特征贡献较均衡、群落分布更合理。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）该保留几维与群落观察（集成：累积解释变异 + 维度选择 + 2D 可视化 + 潜在空间直觉）
#   情境：你有一份较多字段的都市建筑数据（楼高、面积、容积率、屋龄、距捷运距离等），
#         含两种用途类别，想为后续聚类做准备。
#   要求：
#     1. 自造约五到六栏、带两个隐含群落的建筑数据（两类别各栏中心略有差异）。
#     2. 标准化后跑 PCA，画累积解释变异曲线，并说明要保留几维才达 90%。
#     3. 取前两个主成分画 2D 散点并依类别上色，讨论潜在空间中两类是否分离。
#   验收：应看到累积曲线指出的维度数，以及 2D 图中两类别大致聚类，为下一步的聚类铺路。
# TODO: 在这里完成


## 小结

- 降维 dimensionality reduction 的动机：特征太多会带来维度灾难、特征冗余与无法可视化；许多栏其实高度相关，可用更少方向描述。
- 主成分 principal component 是数据散得最开的方向：第一主成分抓最大变异，后续主成分在正交方向抓剩余变异，投影到主成分即得降维座标。
- PCA 看变异量，会被大尺度特征主导，所以做 PCA 前要先用标准化（z-score / StandardScaler）让各栏尺度可比较。
- sklearn 的 PCA 用 fit 学主成分方向、transform 投影得到新座标，n_components 决定降到几维，流程与其他估计器一致。
- 解释变异比例 explained_variance_ratio_ 显示每维保留多少信息；把它累加到门槛（如 90%）可决定该保留几维。
- 把数据压到前两个主成分即可用一张 2D 散点图观察群落与类别分离，是探索数据与为聚类做准备的利器。